# Early Exit: Pretrained ViT-B/16 vs ResNet18 on CIFAR-10

Upgraded from scratch ViT → **Pretrained ViT-B/16** (ImageNet weights) to close the accuracy gap with ResNet18:
- Input resized to **224×224** (ViT-B/16 native resolution)
- **12 Transformer encoder blocks** (embed_dim=768, heads=12)
- Early exit heads after **block 4, block 8, block 12**
- Warm-up (heads only) → full fine-tune strategy
- **Early Exit ResNet18** unchanged as comparison baseline

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights
import matplotlib.pyplot as plt
import numpy as np

# ── drop-in replacement for thop.profile ──────────────────────────────────────
def profile(model, inputs, verbose=False):
    """Lightweight MACs counter — no thop needed."""
    macs = [0]
    params = sum(p.numel() for p in model.parameters())
    hooks = []
    def make_hook(module):
        def hook(m, inp, out):
            if isinstance(m, nn.Conv2d):
                out_h, out_w = out.shape[2], out.shape[3]
                kH, kW = m.kernel_size
                macs[0] += (m.in_channels // m.groups) * kH * kW * m.out_channels * out_h * out_w
            elif isinstance(m, nn.Linear):
                macs[0] += m.in_features * m.out_features
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                macs[0] += inp[0].numel()
            elif isinstance(m, nn.AdaptiveAvgPool2d):
                macs[0] += inp[0].numel()
        return hook
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear, nn.BatchNorm2d, nn.LayerNorm, nn.AdaptiveAvgPool2d)):
            hooks.append(m.register_forward_hook(make_hook(m)))
    model.eval()
    with torch.no_grad():
        model(*inputs)
    for h in hooks:
        h.remove()
    if verbose:
        print(f'MACs: {macs[0]:,} | Params: {params:,}')
    return macs[0], params
# ──────────────────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ✅ Data loaders
# ViT-B/16 requires 224x224 — updated from original 32x32
transform_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=28),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),   # ImageNet mean
                         (0.229, 0.224, 0.225))    # ImageNet std
])
transform_vit_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

# ResNet: 224x224 (unchanged)
transform_rn = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
transform_rn_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_vit  = torchvision.datasets.CIFAR10('./data', train=True,  download=True,  transform=transform_vit)
test_vit   = torchvision.datasets.CIFAR10('./data', train=False, download=False, transform=transform_vit_test)
train_rn   = torchvision.datasets.CIFAR10('./data', train=True,  download=False, transform=transform_rn)
test_rn    = torchvision.datasets.CIFAR10('./data', train=False, download=False, transform=transform_rn_test)

# Reduced batch size for ViT-B/16 (larger model + larger inputs)
train_loader_vit = torch.utils.data.DataLoader(train_vit, batch_size=64,  shuffle=True,  num_workers=2, pin_memory=True)
test_loader_vit  = torch.utils.data.DataLoader(test_vit,  batch_size=64,  shuffle=False, num_workers=2, pin_memory=True)
train_loader_rn  = torch.utils.data.DataLoader(train_rn,  batch_size=64,  shuffle=True,  num_workers=2, pin_memory=True)
test_loader_rn   = torch.utils.data.DataLoader(test_rn,   batch_size=64,  shuffle=False, num_workers=2, pin_memory=True)

print('Data ready.')

---
## Model 1 — Early Exit Pretrained ViT-B/16

**Backbone:** `vit_b_16` with ImageNet pretrained weights  
**Embed dim:** 768 | **Heads:** 12 | **Blocks:** 12  
**Exits:** after block 4 (early), block 8 (mid), block 12 (final) — each uses the `[CLS]` token  
**Strategy:** freeze backbone → warm-up heads for 5 epochs → unfreeze all → fine-tune 20 epochs

In [ ]:
class EarlyExitViT(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        base = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        embed_dim = 768

        # Reuse pretrained components
        self.patch_embedding = base.conv_proj
        self.class_token     = base.class_token
        self.pos_embedding   = base.encoder.pos_embedding
        self.pos_drop        = base.encoder.dropout
        self.blocks          = base.encoder.layers   # 12 TransformerEncoderLayer blocks
        self.ln              = base.encoder.ln

        # Early exit heads after block 4, 8, 12
        self.exit1 = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))
        self.exit2 = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))
        self.exit3 = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B = x.shape[0]
        # Patch embed: (B,3,224,224) -> (B,768,14,14) -> (B,196,768)
        x = self.patch_embedding(x).flatten(2).transpose(1, 2)
        # Prepend CLS token and add positional embedding
        cls = self.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embedding)

        o1, o2, o3 = None, None, None
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i == 3:    o1 = self.exit1(x[:, 0])   # after block 4
            elif i == 7:  o2 = self.exit2(x[:, 0])   # after block 8
            elif i == 11: o3 = self.exit3(x[:, 0])   # after block 12

        return o1, o2, o3


vit_model = EarlyExitViT(num_classes=10).to(device)
total_vit = sum(p.numel() for p in vit_model.parameters())
print(f'Early Exit Pretrained ViT-B/16 total params: {total_vit:,}')

---
## Model 2 — Early Exit ResNet18 (unchanged)

In [ ]:
class EarlyExitResNet(nn.Module):
    def __init__(self, base_model, num_classes=10):
        super().__init__()
        self.stem   = nn.Sequential(base_model.conv1, base_model.bn1,
                                     base_model.relu,  base_model.maxpool)
        self.layer1 = base_model.layer1
        self.layer2 = base_model.layer2
        self.layer3 = base_model.layer3
        self.layer4 = base_model.layer4

        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
        self.exit2 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
        self.final_exit = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x  = self.stem(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return self.exit1(x1), self.exit2(x2), self.final_exit(x4)


base = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in base.parameters():        p.requires_grad = False
for p in base.layer3.parameters(): p.requires_grad = True
for p in base.layer4.parameters(): p.requires_grad = True

resnet_model = EarlyExitResNet(base).to(device)
trainable_rn = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total_rn     = sum(p.numel() for p in resnet_model.parameters())
print(f'Early Exit ResNet total params:     {total_rn:,}')
print(f'Early Exit ResNet trainable params: {trainable_rn:,}')

---
## Shared Utilities

In [ ]:
LOSS_WEIGHTS = (0.5, 0.3, 0.2)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    w1, w2, w3 = LOSS_WEIGHTS
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        o1, o2, o3 = model(inputs)
        loss = w1*criterion(o1, labels) + w2*criterion(o2, labels) + w3*criterion(o3, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, threshold, device):
    model.eval()
    correct, total = 0, 0
    exit_counts  = [0, 0, 0]
    exit_correct = [0, 0, 0]
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            o1, o2, o3 = model(inputs)
            for i in range(inputs.size(0)):
                c1, p1 = torch.max(F.softmax(o1[i], dim=0), dim=0)
                if c1.item() >= threshold:
                    pred, eidx = p1, 0
                else:
                    c2, p2 = torch.max(F.softmax(o2[i], dim=0), dim=0)
                    if c2.item() >= threshold:
                        pred, eidx = p2, 1
                    else:
                        _, p3 = torch.max(F.softmax(o3[i], dim=0), dim=0)
                        pred, eidx = p3, 2
                ok = int(pred.item() == labels[i].item())
                correct += ok
                exit_correct[eidx] += ok
                exit_counts[eidx]  += 1
                total += 1
    return correct / total * 100, exit_counts, exit_correct

print('Utilities ready.')

---
## ViT Warm-up Phase (heads only, 5 epochs)

Freeze the pretrained backbone and train only the 3 exit heads first.
This prevents destroying ImageNet representations before fine-tuning.

In [ ]:
# ── Freeze backbone, train only exit heads ────────────────────────────────────
for name, param in vit_model.named_parameters():
    param.requires_grad = ('exit1' in name or 'exit2' in name or 'exit3' in name)

opt_warmup = optim.Adam(
    filter(lambda p: p.requires_grad, vit_model.parameters()), lr=1e-3
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

WARMUP_EPOCHS = 5
TRAIN_THRESH  = 0.80

print('=== ViT Warm-up (exit heads only) ===')
print(f'{"Epoch":>6}  {"Loss":>10}  {"Acc":>8}')
print('='*32)

for epoch in range(WARMUP_EPOCHS):
    loss_v = train_one_epoch(vit_model, train_loader_vit, criterion, opt_warmup, device)
    acc_v, _, _ = evaluate(vit_model, test_loader_vit, TRAIN_THRESH, device)
    print(f'{epoch+1:>6}  {loss_v:>10.4f}  {acc_v:>7.2f}%')

print('Warm-up complete.')

---
## Train Both Models (25 epochs full fine-tune)

Unfreeze all ViT parameters and fine-tune with a low learning rate.
ResNet training is unchanged from the original notebook.

In [ ]:
NUM_EPOCHS = 25

# Unfreeze all ViT params for full fine-tuning
for param in vit_model.parameters():
    param.requires_grad = True

opt_vit    = optim.AdamW(vit_model.parameters(), lr=1e-4, weight_decay=0.01)
opt_resnet = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-6)

# Cosine schedule for ViT, StepLR for ResNet
sched_vit    = optim.lr_scheduler.CosineAnnealingLR(opt_vit,    T_max=NUM_EPOCHS)
sched_resnet = optim.lr_scheduler.StepLR(opt_resnet, step_size=10, gamma=0.5)

hist_vit    = {'loss': [], 'acc': [], 'exits': []}
hist_resnet = {'loss': [], 'acc': [], 'exits': []}

print('='*65)
print(f'{"Epoch":>5}  {"ViT Loss":>10}  {"ViT Acc":>8}  {"RN Loss":>8}  {"RN Acc":>7}')
print('='*65)

for epoch in range(NUM_EPOCHS):
    # ViT: gradient clipping to keep fine-tuning stable
    vit_model.train()
    total_loss_v = 0.0
    w1, w2, w3 = LOSS_WEIGHTS
    for inputs, labels in train_loader_vit:
        inputs, labels = inputs.to(device), labels.to(device)
        o1, o2, o3 = vit_model(inputs)
        loss = w1*criterion(o1, labels) + w2*criterion(o2, labels) + w3*criterion(o3, labels)
        opt_vit.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)
        opt_vit.step()
        total_loss_v += loss.item()
    loss_v = total_loss_v / len(train_loader_vit)
    acc_v, exits_v, _ = evaluate(vit_model, test_loader_vit, TRAIN_THRESH, device)
    sched_vit.step()

    # ResNet: unchanged
    loss_r = train_one_epoch(resnet_model, train_loader_rn, criterion, opt_resnet, device)
    acc_r, exits_r, _ = evaluate(resnet_model, test_loader_rn, TRAIN_THRESH, device)
    sched_resnet.step()

    hist_vit['loss'].append(loss_v);    hist_vit['acc'].append(acc_v);    hist_vit['exits'].append(exits_v)
    hist_resnet['loss'].append(loss_r); hist_resnet['acc'].append(acc_r); hist_resnet['exits'].append(exits_r)

    print(f'{epoch+1:>5}  {loss_v:>10.4f}  {acc_v:>7.2f}%  {loss_r:>8.4f}  {acc_r:>6.2f}%')

print('='*65)
print('Training complete.')

---
## Threshold Sweep

In [ ]:
thresholds = [0.99, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50]
sweep_vit    = {}
sweep_resnet = {}

print('ViT Threshold Sweep')
print('-'*65)
for thr in thresholds:
    acc, exits, ecorr = evaluate(vit_model, test_loader_vit, thr, device)
    sweep_vit[thr] = {'acc': acc, 'exits': exits, 'ecorr': ecorr}
    print(f'  thr={thr:.2f} | Acc={acc:.2f}% | E1={exits[0]:5d} ({exits[0]/100:.1f}%) | E2={exits[1]:5d} ({exits[1]/100:.1f}%) | E3={exits[2]:5d}')

print('\nResNet Threshold Sweep')
print('-'*65)
for thr in thresholds:
    acc, exits, ecorr = evaluate(resnet_model, test_loader_rn, thr, device)
    sweep_resnet[thr] = {'acc': acc, 'exits': exits, 'ecorr': ecorr}
    print(f'  thr={thr:.2f} | Acc={acc:.2f}% | E1={exits[0]:5d} ({exits[0]/100:.1f}%) | E2={exits[1]:5d} ({exits[1]/100:.1f}%) | E3={exits[2]:5d}')

---
## FLOPs Analysis

In [ ]:
dummy_vit    = torch.randn(1, 3, 224, 224).to(device)  # ViT-B/16 uses 224x224
dummy_resnet = torch.randn(1, 3, 224, 224).to(device)

# ViT exit wrappers
class ViTExit1(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        B = x.shape[0]
        x = self.m.patch_embedding(x).flatten(2).transpose(1, 2)
        cls = self.m.class_token.expand(B, -1, -1)
        x = self.m.pos_drop(torch.cat([cls, x], dim=1) + self.m.pos_embedding)
        for i, block in enumerate(self.m.blocks):
            x = block(x)
            if i == 3: return self.m.exit1(x[:, 0])

class ViTExit2(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        B = x.shape[0]
        x = self.m.patch_embedding(x).flatten(2).transpose(1, 2)
        cls = self.m.class_token.expand(B, -1, -1)
        x = self.m.pos_drop(torch.cat([cls, x], dim=1) + self.m.pos_embedding)
        for i, block in enumerate(self.m.blocks):
            x = block(x)
            if i == 7: return self.m.exit2(x[:, 0])

class ViTExit3(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        B = x.shape[0]
        x = self.m.patch_embedding(x).flatten(2).transpose(1, 2)
        cls = self.m.class_token.expand(B, -1, -1)
        x = self.m.pos_drop(torch.cat([cls, x], dim=1) + self.m.pos_embedding)
        for i, block in enumerate(self.m.blocks):
            x = block(x)
            if i == 11: return self.m.exit3(x[:, 0])

vf1, _ = profile(ViTExit1(vit_model).to(device), inputs=(dummy_vit,), verbose=False)
vf2, _ = profile(ViTExit2(vit_model).to(device), inputs=(dummy_vit,), verbose=False)
vf3, _ = profile(ViTExit3(vit_model).to(device), inputs=(dummy_vit,), verbose=False)

# ResNet exit wrappers (unchanged)
class RNExit1(nn.Module):
    def __init__(self, m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.e=m.exit1
    def forward(self, x): return self.e(self.l1(self.stem(x)))
class RNExit2(nn.Module):
    def __init__(self, m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.l2=m.layer2; self.e=m.exit2
    def forward(self, x): return self.e(self.l2(self.l1(self.stem(x))))
class RNExit3(nn.Module):
    def __init__(self, m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.l2=m.layer2; self.l3=m.layer3; self.l4=m.layer4; self.e=m.final_exit
    def forward(self, x): return self.e(self.l4(self.l3(self.l2(self.l1(self.stem(x))))))

rf1, _ = profile(RNExit1(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)
rf2, _ = profile(RNExit2(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)
rf3, _ = profile(RNExit3(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)

print('FLOPs per exit point')
print('-'*52)
for tag, f1, f2, f3 in [('ViT   ', vf1, vf2, vf3), ('ResNet', rf1, rf2, rf3)]:
    print(f'  {tag} Exit1: {f1/1e6:7.2f} MFLOPs  (saves {(1-f1/f3)*100:.1f}%)')
    print(f'  {tag} Exit2: {f2/1e6:7.2f} MFLOPs  (saves {(1-f2/f3)*100:.1f}%)')
    print(f'  {tag} Exit3: {f3/1e6:7.2f} MFLOPs  (full)')
    print()

In [ ]:
# Expected avg FLOPs per sample at each threshold
vit_flops    = [vf1, vf2, vf3]
resnet_flops = [rf1, rf2, rf3]

print(f'  {"Threshold":>10} | {"ViT MFLOPs":>11} | {"ViT Save":>9} | {"ViT Acc":>8} | {"RN MFLOPs":>10} | {"RN Save":>8} | {"RN Acc":>7}')
print('-'*80)
for thr in thresholds:
    ve  = sweep_vit[thr]['exits'];    avg_v = sum(c*f for c,f in zip(ve,  vit_flops))    / sum(ve)
    re  = sweep_resnet[thr]['exits']; avg_r = sum(c*f for c,f in zip(re,  resnet_flops)) / sum(re)
    print(f'  {thr:>10.2f} | {avg_v/1e6:>11.2f} | {(1-avg_v/vf3)*100:>8.1f}% | {sweep_vit[thr]["acc"]:>7.2f}% | {avg_r/1e6:>10.2f} | {(1-avg_r/rf3)*100:>7.1f}% | {sweep_resnet[thr]["acc"]:>6.2f}%')

---
## Visualizations

In [ ]:
epochs   = range(1, NUM_EPOCHS + 1)
thr_vals = thresholds

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Early Exit: Pretrained ViT-B/16 vs ResNet18 on CIFAR-10',
             fontsize=14, fontweight='bold')

# Training Loss
axes[0,0].plot(epochs, hist_vit['loss'],    label='ViT-B/16', color='mediumpurple', lw=2)
axes[0,0].plot(epochs, hist_resnet['loss'], label='ResNet18', color='tomato',       lw=2)
axes[0,0].set_title('Training Loss'); axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Accuracy over epochs
axes[0,1].plot(epochs, hist_vit['acc'],    label='ViT-B/16', color='mediumpurple', lw=2)
axes[0,1].plot(epochs, hist_resnet['acc'], label='ResNet18', color='tomato',       lw=2)
axes[0,1].set_title(f'Test Accuracy (thr={TRAIN_THRESH})')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Accuracy vs threshold
axes[0,2].plot(thr_vals, [sweep_vit[t]['acc']    for t in thr_vals], 'o-', label='ViT-B/16', color='mediumpurple', lw=2)
axes[0,2].plot(thr_vals, [sweep_resnet[t]['acc'] for t in thr_vals], 's-', label='ResNet18', color='tomato',       lw=2)
axes[0,2].set_title('Accuracy vs Threshold')
axes[0,2].set_xlabel('Confidence Threshold'); axes[0,2].set_ylabel('Accuracy (%)')
axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

# ViT exit distribution stackplot
e1v = [e[0] for e in hist_vit['exits']]
e2v = [e[1] for e in hist_vit['exits']]
e3v = [e[2] for e in hist_vit['exits']]
axes[1,0].stackplot(epochs, e1v, e2v, e3v,
                    labels=['Exit1 (blk4)','Exit2 (blk8)','Exit3 (blk12)'],
                    colors=['#9b59b6','#3498db','#1abc9c'], alpha=0.8)
axes[1,0].set_title('ViT-B/16 Exit Distribution per Epoch')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Samples')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# ResNet exit distribution stackplot
e1r = [e[0] for e in hist_resnet['exits']]
e2r = [e[1] for e in hist_resnet['exits']]
e3r = [e[2] for e in hist_resnet['exits']]
axes[1,1].stackplot(epochs, e1r, e2r, e3r,
                    labels=['Exit1','Exit2','Exit3'],
                    colors=['#2ecc71','#f39c12','#e74c3c'], alpha=0.8)
axes[1,1].set_title('ResNet18 Exit Distribution per Epoch')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Samples')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

# FLOPs savings vs threshold
vit_sav = [(1 - sum(c*f for c,f in zip(sweep_vit[t]['exits'],    vit_flops))    / sum(sweep_vit[t]['exits'])    / vf3)*100 for t in thr_vals]
rn_sav  = [(1 - sum(c*f for c,f in zip(sweep_resnet[t]['exits'], resnet_flops)) / sum(sweep_resnet[t]['exits']) / rf3)*100 for t in thr_vals]
axes[1,2].plot(thr_vals, vit_sav, 'o-', label='ViT-B/16', color='mediumpurple', lw=2)
axes[1,2].plot(thr_vals, rn_sav,  's-', label='ResNet18', color='tomato',       lw=2)
axes[1,2].set_title('FLOPs Savings vs Threshold')
axes[1,2].set_xlabel('Confidence Threshold'); axes[1,2].set_ylabel('FLOPs Saved (%)')
axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('vit_vs_resnet_early_exit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as vit_vs_resnet_early_exit.png')

In [ ]:
# ✅ Save models
torch.save(vit_model.state_dict(),    'early_exit_vit_cifar10.pth')
torch.save(resnet_model.state_dict(), 'early_exit_resnet_cifar10.pth')
print('Models saved.')